# 🔄 VILAW-LLM: CONTINUAL TRAINING & DATASET INTEGRATION
### Tự động tải các nguồn dữ liệu mới, làm sạch theo chuẩn Pháp lý & Huấn luyện tiếp nối từ `vilaw-dpo-lora`

> **Chiến lược dữ liệu Continual Training:**
> 1. **`duyet/vietnamese-legal-instruct`** (~222K mẫu): Nguồn pháp lý mở rộng cực lớn, trích xuất & lọc lấy các mẫu có căn cứ pháp luật rõ ràng.
> 2. **`hoanghai2110/vietnamese-dataset`** (~2K mẫu): Bổ trợ độ tự nhiên, phản xạ giao tiếp tiếng Việt trôi chảy.
> 3. **Pipeline làm sạch & lọc chuẩn pháp lý:** Chuẩn hoá Unicode NFC, lọc bỏ câu quá ngắn hoặc thiếu viện dẫn điều khoản.
> 4. **Continual Fine-Tuning:** Nạp model đã align DPO `vilaw-dpo-lora`, mở khóa LoRA và train với Learning Rate nhỏ ($5 \times 10^{-5}$) để bảo toàn năng lực viện dẫn luật.
> 5. **Xuất bản:** Đóng gói thành `vilaw-dpo-lora-v2.zip`.

## 1. Kiểm tra GPU & Cài đặt Thư viện Cần thiết

In [ ]:
!nvidia-smi

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.29" trl peft accelerate bitsandbytes
!pip install pyarrow pandas datasets scikit-learn

## 2. Giải nén & Nạp Model `vilaw-dpo-lora` Hiện Có

In [ ]:
import os

# 2.1. Tự động giải nén nếu bạn tải lên vilaw-dpo-lora.zip
zip_files = [f for f in os.listdir('.') if f.endswith('.zip') and 'dpo' in f.lower()]
if zip_files:
    target_zip = zip_files[0]
    print(f"-> Đang giải nén: {target_zip}...")
    !mkdir -p vilaw-dpo-lora
    !unzip -o -q "$target_zip" -d vilaw-dpo-lora
    print("✓ Giải nén hoàn tất!")

# 2.2. Dò tìm thư mục chứa adapter_config.json của DPO model
adapter_dir = None
for root, dirs, files in os.walk("."):
    if "adapter_config.json" in files and "dpo" in root.lower():
        adapter_dir = root
        break
if adapter_dir is None:
    for root, dirs, files in os.walk("."):
        if "adapter_config.json" in files:
            adapter_dir = root
            break

print(f"🎯 ĐÃ TÌM THẤY DPO ADAPTER TẠI: '{adapter_dir}'")

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048

print(f"Đang nạp mô hình từ checkpoint: {adapter_dir}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=adapter_dir,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

# Mở khóa các tham số LoRA để tiếp tục huấn luyện (requires_grad = True)
for name, param in model.named_parameters():
    if "lora" in name.lower():
        param.requires_grad = True

print("✓ Nạp mô hình DPO thành công và đã mở khóa LoRA để Continual Training!")

## 3. Tải Tự Động & Tiền Xử Lý Các Nguồn Dữ Liệu Mới
Cell này sẽ tự động tải các dataset chưa dùng từ Hugging Face:
- `duyet/vietnamese-legal-instruct`: Stream lấy mẫu chất lượng cao, lọc các cặp hỏi-đáp pháp lý.
- `hoanghai2110/vietnamese-dataset`: Bổ trợ phản xạ giao tiếp tự nhiên bằng tiếng Việt.
- Nếu bạn có file riêng `new_dataset.parquet`/`csv`, cell cũng sẽ tự động tích hợp gộp chung.

In [ ]:
import re
import unicodedata
import pandas as pd
from datasets import load_dataset, Dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="chatml")

SYSTEM_PROMPT = (
    "Bạn là một chuyên gia tư vấn pháp luật Việt Nam am hiểu sâu sắc các quy định pháp luật. "
    "Hãy trả lời câu hỏi dựa trên các văn bản quy phạm pháp luật hiện hành, "
    "viện dẫn chính xác số Điều, Khoản, tên luật và đưa ra lập luận logic, rõ ràng."
)

# 3.1. Hàm chuẩn hoá Unicode NFC & loại bỏ khoảng trắng thừa
def normalize_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize('NFC', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

LEGAL_KEYWORDS = ["điều", "khoản", "điểm", "luật", "nghị định", "thông tư", "quy định", "bộ luật", "pháp lệnh"]

collected_samples = []

# 3.2. Nguồn 1: duyet/vietnamese-legal-instruct (Streaming để tối ưu RAM)
print("📥 [1/3] Đang tải & lọc từ duyet/vietnamese-legal-instruct...")
try:
    ds_duyet = load_dataset("duyet/vietnamese-legal-instruct", split="train", streaming=True)
    count_duyet = 0
    TARGET_DUYET = 8000  # Lấy 8,000 mẫu chuẩn pháp lý cao cấp nhất
    for item in ds_duyet:
        convs = item.get("conversations", [])
        user_msg = ""
        assistant_msg = ""
        for c in convs:
            if c.get("role") == "user":
                user_msg = normalize_text(c.get("content", ""))
            elif c.get("role") == "assistant":
                assistant_msg = normalize_text(c.get("content", ""))
        
        # Quality Filter: Đủ độ dài & có từ khóa căn cứ luật
        if len(user_msg) >= 15 and len(assistant_msg) >= 50:
            if any(kw in assistant_msg.lower() for kw in LEGAL_KEYWORDS):
                collected_samples.append({
                    "user": user_msg,
                    "assistant": assistant_msg,
                    "source": "duyet_legal_instruct"
                })
                count_duyet += 1
                if count_duyet >= TARGET_DUYET:
                    break
    print(f"   ✓ Đã thu thập {count_duyet:,} mẫu pháp lý từ duyet/vietnamese-legal-instruct")
except Exception as e:
    print(f"   ⚠️ Không thể tải duyet/vietnamese-legal-instruct: {e}")

# 3.3. Nguồn 2: hoanghai2110/vietnamese-dataset (Bổ trợ độ tự nhiên tiếng Việt)
print("📥 [2/3] Đang tải từ hoanghai2110/vietnamese-dataset...")
try:
    ds_general = load_dataset("hoanghai2110/vietnamese-dataset", split="train")
    count_gen = 0
    for item in ds_general:
        msgs = item.get("messages", [])
        u_msg, a_msg = "", ""
        for m in msgs:
            if m.get("role") == "user":
                u_msg = normalize_text(m.get("content", ""))
            elif m.get("role") == "assistant":
                a_msg = normalize_text(m.get("content", ""))
        if len(u_msg) >= 10 and len(a_msg) >= 20:
            collected_samples.append({
                "user": u_msg,
                "assistant": a_msg,
                "source": "hoanghai_vietnamese"
            })
            count_gen += 1
    print(f"   ✓ Đã thu thập {count_gen:,} mẫu giao tiếp tự nhiên từ hoanghai2110/vietnamese-dataset")
except Exception as e:
    print(f"   ⚠️ Không thể tải hoanghai2110/vietnamese-dataset: {e}")

# 3.4. Nguồn 3: File local nếu người dùng có upload thêm (new_dataset.*)
print("📥 [3/3] Quét file dữ liệu tải lên thêm (nếu có)...")
for candidate in ["new_dataset.parquet", "new_dataset.csv", "new_dataset.jsonl"]:
    if os.path.exists(candidate):
        print(f"   -> Phát hiện file local: {candidate}")
        df_local = pd.read_parquet(candidate) if candidate.endswith('.parquet') else pd.read_csv(candidate)
        q_c = "question" if "question" in df_local.columns else df_local.columns[0]
        a_c = "answer" if "answer" in df_local.columns else df_local.columns[1]
        for _, r in df_local.iterrows():
            q_t = normalize_text(str(r[q_c]))
            a_t = normalize_text(str(r[a_c]))
            if q_t and a_t:
                collected_samples.append({"user": q_t, "assistant": a_t, "source": "uploaded_file"})
        break

# 3.5. Deduplication & Format sang ChatML
df_all = pd.DataFrame(collected_samples)
initial_len = len(df_all)
df_all = df_all.drop_duplicates(subset=["user"])
print(f"\n📊 TỔNG KẾT DỮ LIỆU HUẤN LUYỆN:")
print(f"   - Thu thập ban đầu: {initial_len:,} mẫu")
print(f"   - Sau khi Dedup:    {len(df_all):,} mẫu sạch chất lượng cao!")
print(f"   - Phân bổ nguồn:    {df_all['source'].value_counts().to_dict()}")

formatted_convos = []
for _, row in df_all.iterrows():
    formatted_convos.append({
        "conversations": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": row["user"]},
            {"role": "assistant", "content": row["assistant"]}
        ]
    })

train_dataset = Dataset.from_pandas(pd.DataFrame(formatted_convos))

def apply_chatml(examples):
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in examples["conversations"]
    ]
    return {"text": texts}

train_dataset = train_dataset.map(apply_chatml, batched=True)
print("✓ Toàn bộ dữ liệu đã được ánh xạ chuẩn ChatML và sẵn sàng huấn luyện!")

## 4. Huấn Luyện Tiếp Nối (Continual Fine-Tuning)
- **Learning Rate:** $5 \times 10^{-5}$ (tốc độ học an toàn để không làm mất căn chỉnh DPO).
- **Batch Size:** 2 (gradient_accumulation=8 $\rightarrow$ Effective batch = 16).
- **Scheduler:** Cosine decay với warmup 5%.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./vilaw-v2-checkpoints",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,                 # Tốc độ học an toàn cho continual training
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    optim="adamw_8bit",
    fp16=True,
    logging_steps=10,
    num_train_epochs=1,                # 1 epoch trên tập dữ liệu mở rộng
    max_steps=350,                     # 350 steps (~5.600 samples) tối ưu cho 1 phiên Colab T4
    save_strategy="no",
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=False,
    args=training_args,
)

print("🚀 Bắt đầu quá trình Continual Training...")
trainer_stats = trainer.train()

## 5. Kiểm thử Suy luận & Xuất Bản Adapter `vilaw-dpo-lora-v2`

In [ ]:
# 5.1. Chuyển sang chế độ Inference
FastLanguageModel.for_inference(model)

test_query = "Người lao động đơn phương chấm dứt hợp đồng lao động trái pháp luật thì có được nhận trợ cấp thôi việc không? Viện dẫn luật cụ thể."

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": test_query}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=512,
    temperature=0.3,
    repetition_penalty=1.15
)

tra_loi = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print("=== CÂU TRẢ LỜI CỦA VILAW-LLM V2 ===\n")
print(tra_loi)

# 5.2. Lưu Adapter mới v2 & Đóng gói zip
output_dir = "vilaw-dpo-lora-v2"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
!zip -r vilaw-dpo-lora-v2.zip vilaw-dpo-lora-v2
print(f"\n✅ HOÀN TẤT! File '{output_dir}.zip' đã sẵn sàng để tải về!")